# Collections Recovery Audit - Final Reproducible Notebook

Purpose: independently reconstruct collections recovery, test the reported 11% improvement, investigate data-quality and attribution failures, evaluate drivers and bias, design a counterfactual, and size the ₹10 Cr decision.

**Evidence labels:** Fact / Strong Evidence / Correlation / Hypothesis.


In [1]:
from pathlib import Path
import pandas as pd, numpy as np
RAW=Path('/mnt/data/raw_dataset')
def r(f): return pd.read_csv(RAW/f,low_memory=False)
A=r('accounts.csv'); B=r('borrowers.csv'); AG=r('agents.csv'); SES=r('agent_sessions.csv')
C=r('calls.csv'); CD=r('call_dispositions.csv'); AT=r('call_attempts.csv'); T=r('daily_targeting.csv')
CAMP=r('campaigns.csv'); PTP=r('promises_to_pay.csv'); P=r('payments.csv'); V=r('vendor_telephony.csv')
for df,col in [(P,'event_at'),(C,'event_at'),(CD,'event_at'),(AT,'event_at'),(T,'target_date'),(PTP,'event_at'),(SES,'login_at'),(SES,'logout_at'),(A,'opened_at'),(AG,'joined_at')]:
    df[col]=pd.to_datetime(df[col],errors='coerce')
print('Rows:',{k:len(v) for k,v in {'accounts':A,'borrowers':B,'agents':AG,'calls':C,'payments':P,'targeting':T}.items()})


Rows: {'accounts': 30000, 'borrowers': 30600, 'agents': 30000, 'calls': 91350, 'payments': 25500, 'targeting': 45000}


## 1. Source coverage and analytical grain

The supplied event data run from 1 Jan 2026 through 8 Aug 2026. August is partial, so Jan-Jul are the complete month trend window.

`account_id` is the analytical event spine. The supplied event-level borrower IDs are not reliable enough for historical joins.


## 2. Golden payment ledger and recovery definition

In [2]:
P1=P.sort_values(['payment_id','event_at']).drop_duplicates('payment_id')
S=P1[P1.payment_status.eq('SUCCESS')].copy()
S['month']=S.event_at.dt.to_period('M').astype(str)
monthly=S.groupby('month').agg(recovery=('amount','sum'),payers=('account_id','nunique'),payments=('payment_id','count'))
monthly['recovery_per_payer']=monthly.recovery/monthly.payers
monthly['mom_cash_pct']=monthly.recovery.pct_change()*100
monthly['mom_payer_pct']=monthly.payers.pct_change()*100
monthly['mom_rpp_pct']=monthly.recovery_per_payer.pct_change()*100
display(monthly.round(3))


,recovery,payers,payments,recovery_per_payer,mom_cash_pct,mom_payer_pct,mom_rpp_pct
month,,,,,,,
2026-01,1.872291e+08,2374,2464,78866.524,NaN,NaN,NaN
2026-02,1.701425e+08,2173,2268,78298.414,-9.126,-8.467,-0.720
2026-03,1.889124e+08,2419,2524,78095.235,11.032,11.321,-0.259
2026-04,1.751380e+08,2304,2406,76014.776,-7.291,-4.754,-2.664
2026-05,1.842503e+08,2344,2449,78605.068,5.203,1.736,3.408
2026-06,1.755597e+08,2286,2366,76797.781,-4.717,-2.474,-2.299
2026-07,1.872423e+08,2335,2441,80189.407,6.654,2.143,4.416
2026-08,4.710970e+07,608,616,77483.051,-74.840,-73.961,-3.375


In [3]:
feb,mar=monthly.loc['2026-02'],monthly.loc['2026-03']
print('Fact - cash change:',round((mar.recovery/feb.recovery-1)*100,2),'%')
print('Strong Evidence - payer change:',round((mar.payers/feb.payers-1)*100,2),'%')
print('Strong Evidence - recovery/payer change:',round((mar.recovery_per_payer/feb.recovery_per_payer-1)*100,2),'%')


Fact - cash change: 11.03 %
Strong Evidence - payer change: 11.32 %
Strong Evidence - recovery/payer change: -0.26 %


## 3. Operational metric framework

- Contact rate = distinct accounts with ANSWERED call / distinct accounts attempted.
- RPC rate = distinct accounts with RPC-like disposition on ANSWERED calls / distinct answered accounts.
- PTP kept rate = KEPT / (KEPT + BROKEN + CANCELLED); OPEN is unresolved.
- Recovery per agent-hour = clean recovery / collection session hours. This is productivity, not causal agent attribution.
- Cost per ₹ recovered is **not computable** because the supplied package has no labor, telephony, messaging, vendor or field cost ledger.
- Historical recovery rate vs opening outstanding is **not reliable** because historical periodized opening balances are not supplied.
- Channel conversion is descriptive 14-day post-exposure conversion, not causal ROI.


In [4]:
C1=C.sort_values(['call_id','event_at']).drop_duplicates('call_id').copy()
C1['month']=C1.event_at.dt.to_period('M').astype(str)
D=CD.merge(C1[['call_id','call_status','month']],on='call_id',how='left')
D['rpc_like']=D.disposition_code.isin(['PTP','PROMISE_TO_PAY','PAID','CALLBACK'])
ops=C1.groupby('month').agg(attempted=('account_id','nunique'))
ops['answered']=C1[C1.call_status.eq('ANSWERED')].groupby('month').account_id.nunique()
ops['rpc']=D[D.call_status.eq('ANSWERED')&D.rpc_like].groupby('month').account_id.nunique()
ops['contact_rate']=ops.answered/ops.attempted
ops['rpc_rate']=ops.rpc/ops.answered
PTP['month']=PTP.event_at.dt.to_period('M').astype(str)
pt=PTP.groupby('month').status.agg(kept=lambda s:(s=='KEPT').sum(),broken=lambda s:(s=='BROKEN').sum(),cancelled=lambda s:(s=='CANCELLED').sum())
pt['ptp_kept_rate']=pt.kept/(pt.kept+pt.broken+pt.cancelled)
SES['hours']=((SES.logout_at-SES.login_at).dt.total_seconds()/3600).clip(lower=0)
SES['month']=SES.login_at.dt.to_period('M').astype(str)
ops=ops.join(pt[['ptp_kept_rate']]).join(SES.groupby('month').hours.sum().rename('agent_hours')).join(monthly[['recovery']])
ops['recovery_per_agent_hour']=ops.recovery/ops.agent_hours
display(ops.round(4))


,attempted,answered,rpc,contact_rate,rpc_rate,ptp_kept_rate,agent_hours,recovery,recovery_per_agent_hour
month,,,,,,,,,
2025-12,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-01,10324,2433.0,410.0,0.2357,0.1685,0.3246,11162.5786,1.872291e+08,16772.9280
2026-02,9532,2188.0,376.0,0.2295,0.1718,0.3395,10557.0114,1.701425e+08,16116.5360
2026-03,10416,2457.0,431.0,0.2359,0.1754,0.3251,11130.8200,1.889124e+08,16972.0087
2026-04,10036,2258.0,425.0,0.2250,0.1882,0.3321,10620.3217,1.751380e+08,16490.8417
2026-05,10370,2491.0,452.0,0.2402,0.1815,0.3339,10833.1292,1.842503e+08,17008.0386
2026-06,9971,2385.0,474.0,0.2392,0.1987,0.3256,10703.0086,1.755597e+08,16402.8390
2026-07,10278,2345.0,392.0,0.2282,0.1672,0.3267,11180.1586,1.872423e+08,16747.7289
2026-08,3053,619.0,113.0,0.2028,0.1826,0.3543,2683.8597,4.710970e+07,17552.9648


## 4. Data forensics

In [5]:
print('Duplicate payment IDs:',P.groupby('payment_id').size().gt(1).sum())
print('Payment references spanning multiple accounts:',P.groupby('payment_reference').account_id.nunique().gt(1).sum())
print('Agent IDs with conflicting identity fields:',AG.groupby('agent_id')[['employee_code','vendor_id','team','agent_name']].nunique().gt(1).any(axis=1).sum())
print('Duplicate call IDs:',C.groupby('call_id').size().gt(1).sum())
print('RPC-like dispositions on non-ANSWERED calls:',int((D.rpc_like & D.call_status.ne('ANSWERED')).sum()))


Duplicate payment IDs: 500
Payment references spanning multiple accounts: 3407
Agent IDs with conflicting identity fields: 1000
Duplicate call IDs: 1350
RPC-like dispositions on non-ANSWERED calls: 12422


## 5. Drivers

Observable dimensions: loan type, risk, DPD, geography, campaign/channel, telephony vendor, calling hour, attempt intensity, borrower segment proxy and agent tenure.

**Not available in supplied schemas:** client and language. Do not infer them from names or geography.


In [6]:
A2=A.merge(B[['borrower_id','city','state']],on='borrower_id',how='left')
SD=S.merge(A2[['account_id','loan_type','dpd','risk_segment','opened_at','borrower_id','state']],on='account_id',how='left')
SD['dpd_band']=pd.cut(SD.dpd,[-1,7,30,60,90,180,999],labels=['0-7','8-30','31-60','61-90','91-180','181+'])
for d in ['loan_type','risk_segment','dpd_band','state']:
    g=SD.groupby(d,observed=False).agg(recovery=('amount','sum'),payers=('account_id','nunique'))
    g['recovery_per_payer']=g.recovery/g.payers
    print('\n',d); display(g.sort_values('recovery_per_payer',ascending=False).head(8))



 loan_type


,recovery,payers,recovery_per_payer
loan_type,,,
CREDIT_CARD,7.130411e+08,2690,265071.049955
AUTO,7.006667e+08,2700,259506.192459
PERSONAL,6.778402e+08,2629,257831.963648
CONSUMER,6.910729e+08,2685,257382.822644
BNPL,6.598309e+08,2580,255748.409337



 risk_segment


,recovery,payers,recovery_per_payer
risk_segment,,,
HIGH,8.713056e+08,3300,264032.011803
NPA,8.481102e+08,3268,259519.658984
MEDIUM,8.600777e+08,3335,257894.350570
LOW,8.629583e+08,3381,255237.594700



 dpd_band


,recovery,payers,recovery_per_payer
dpd_band,,,
0-7,9.415273e+08,3603,261317.589953
31-60,6.574880e+08,2524,260494.447639
91-180,6.210320e+08,2385,260390.794922
61-90,6.314467e+08,2436,259214.558440
8-30,5.909579e+08,2336,252978.543943
181+,0.000000e+00,0,NaN



 state


,recovery,payers,recovery_per_payer
state,,,
Maharashtra,6.590339e+08,5061,130218.109810
Odisha,3.427979e+08,2946,116360.465377
Delhi,3.511827e+08,3037,115634.745845
Haryana,3.227456e+08,2810,114856.087370
Rajasthan,3.267779e+08,2856,114418.041548
West Bengal,3.261549e+08,2859,114080.076296
Tamil Nadu,3.345739e+08,2938,113878.112267
Telangana,3.318231e+08,2924,113482.603694


## 6. Statistical checks

In [7]:
print('Mix decomposition:'); display(pd.read_csv('/mnt/data/final_submission_v2/data/mix_decomposition.csv'))
print('Simpson sign checks:'); display(pd.read_csv('/mnt/data/final_submission_v2/data/simpson_check.csv'))
print('Cohort effects:'); display(pd.read_csv('/mnt/data/final_submission_v2/data/cohort_effect.csv'))
print('14-day attribution-window diagnostics:'); display(pd.read_csv('/mnt/data/final_submission_v2/data/attribution_window_14d.csv'))


Mix decomposition:


,dimension,march_actual_rpp,march_standardized_rpp,standardized_vs_actual_pct
0,risk_segment,199121.301013,199282.898129,0.081155
1,loan_type,199121.301013,199194.032696,0.036526
2,dpd_band,199121.301013,199032.101657,-0.044796
3,risk_segment+loan_type,199121.301013,199581.270753,0.231000
4,risk_segment+loan_type+dpd_band,199121.301013,200436.159758,0.660331


Simpson sign checks:


,dimension,common_strata,positive_mar_vs_feb,positive_share
0,risk_segment,4,2,0.500000
1,loan_type,5,2,0.400000
2,dpd_band,5,3,0.600000
3,state,9,6,0.666667


Cohort effects:


,month,age_band,recovery,payers,rpp
0,2026-02,<3m,1.045652e+07,57,183447.787018
1,2026-02,3-6m,5.543509e+07,286,193828.982448
2,2026-02,6-12m,1.082277e+08,544,198948.010202
3,2026-02,12m+,2.602526e+08,1291,201589.959876
4,2026-03,<3m,0.000000e+00,0,NaN
5,2026-03,3-6m,5.727127e+07,271,211333.106052
6,2026-03,6-12m,1.219092e+08,623,195680.919775
7,2026-03,12m+,3.024939e+08,1527,198096.884342


14-day attribution-window diagnostics:


,candidate_targets_14d,payments,recovery,recovery_share
0,0.0,15979,1.198776e+09,0.911212
1,1.0,1476,1.108309e+08,0.084245
2,2.0,76,5.663242e+06,0.004305
3,3.0,3,3.140195e+05,0.000239


Interpretation:
- **Mix:** standardization is close to actual March, so observed mix is not the main explanation.
- **Cohort:** age-band results test whether account maturity changed.
- **Selection/survivorship:** historical targeting is not randomized and current account status is overwritten; observational comparisons are not causal.
- **Simpson:** report within-stratum direction checks; do not claim causality.
- **Attribution-window:** multiple/no candidate touches make last-touch ROI unreliable.
- **Time series:** July does not preserve a sustained March step-up.


## 7. Counterfactual

Strategy versions overlap in time, so a historical pre/post targeting estimate is not identified cleanly. Recommended identification strategy: randomize eligible accounts into treatment vs holdout (or a stepped-wedge rollout), stratified by DPD/risk/portfolio and blocked on geography where useful. Primary outcome: incremental 14- and 30-day recovery per eligible account. Secondary outcomes: payer rate, recovery/payer, complaints and contact burden. Use cost per recovered rupee only after a cost ledger is added.

## 8. ₹10 Cr investment case

In [8]:
annualized=monthly.loc[monthly.index<='2026-07','recovery'].sum()/7*12
for lift in [0,.02,.05,.08]:
    inc=annualized*lift
    print(f'lift={lift:.0%} | incremental=₹{inc/1e7:.2f} Cr | benefit/cost={inc/1e8:.2f}x | net ROI={(inc/1e8-1)*100:.1f}%')
print('Break-even lift:',1e8/annualized)


lift=0% | incremental=₹0.00 Cr | benefit/cost=0.00x | net ROI=-100.0%
lift=2% | incremental=₹4.35 Cr | benefit/cost=0.43x | net ROI=-56.5%
lift=5% | incremental=₹10.87 Cr | benefit/cost=1.09x | net ROI=8.7%
lift=8% | incremental=₹17.40 Cr | benefit/cost=1.74x | net ROI=74.0%
Break-even lift: 0.04598700560488675


**Recommendation:** Better Borrower Targeting.

**Planning midpoint:** 5% causal lift. This implies approximately ₹10.87 Cr incremental recovery against ₹10 Cr investment, 1.09x gross benefit/cost and 8.7% net ROI. The 5% number is a planning scenario, not a causal estimate from the current observational data. Confidence is low for causal lift and high for the cleaned payment-ledger reconstruction.

## 9. Reproducibility

All final CSV outputs in `data/` are generated from the supplied raw tables; the production SQL in `sql/` defines the same cleaning and metric logic. The notebook intentionally avoids loading a precomputed attribution result as a substitute for analysis.